# Foundry IQ setup

**Notebook 3 of 4.** Builds the full Foundry IQ stack over the `arxiv-nlp` index:

```
arxiv-nlp index  (Azure AI Search)
    └── KnowledgeSource  (arxiv-nlp-ks)          - registers the index as a retrieval target
            └── KnowledgeBase  (arxiv-nlp-kb-fast)   - minimal effort: no LLM, fastest
            └── KnowledgeBase  (arxiv-nlp-kb)        - low effort: one LLM planning pass
                    MCP endpoints auto-exposed
                        └── MCP Connection  (arxiv-nlp-mcp)       - RemoteTool → arxiv-nlp-kb
                        └── MCP Connection  (arxiv-nlp-mcp-fast)  - RemoteTool → arxiv-nlp-kb-fast
                                └── Foundry Agent  (arxiv-nlp-agent, versioned)
                                        └── MCPTool kb_standard - calls arxiv-nlp-kb (low effort)
                                        └── MCPTool kb_fast     - calls arxiv-nlp-kb-fast (minimal)
```

**Phases:**
1. **Knowledge Source** - registers `arxiv-nlp` index as a retrieval target
2. **Knowledge Bases** - dual configs: `fast` (minimal reasoning) and standard (low reasoning)
3. **Inline KB validation** - direct retrieval test before wiring to an agent
4. **MCP Connections** - one per KB, created via Azure Management REST API
5. **Versioned Agent** - `create_version()` with both KB tools; agent picks based on query complexity

## Prerequisites

1. **Run `10-02-index-and-ingest.ipynb`** - the `arxiv-nlp` index must exist and be populated.
2. **`.env` file** - the following keys must be set (all except `IQ_GATEWAY_KEY` are
   written automatically by `10-01-deploy-search-and-project.ipynb`):
   ```
   IQ_SEARCH_ENDPOINT=https://iq-search-{suffix}.search.windows.net
   IQ_FOUNDRY_PROJECT_ENDPOINT=https://aif-spoke-multi-{suffix}.services.ai.azure.com/api/projects/iq-project
   IQ_APIM_CONNECTION=iq-apim-connection
   IQ_GATEWAY_KEY=<your APIM subscription key>
   GATEWAY_URL=https://apim-foundry-{suffix}.azure-api.net/openai
   CHAT_MODEL=gpt-4.1-mini
   ```
3. **Azure subscription ID** - discoverable via `az account show --query id -o tsv`.
   Set `AZURE_SUBSCRIPTION_ID` in `.env` or the notebook will discover it at runtime.
4. **Resource group name** - the resource group used for the Bicep deployment.
   Set `IQ_RESOURCE_GROUP` in `.env` (e.g. `rg-foundry-multi-{suffix}`).
5. **Python environment** - `uv sync`, select `.venv` kernel.
6. **Azure CLI** - `az login`.

## Imports and configuration

In [1]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

# ── Resource names ──────────────────────────────────────────────────────────
INDEX_NAME     = 'arxiv-nlp'
KS_NAME        = 'arxiv-nlp-ks'
KB_FAST        = 'arxiv-nlp-kb-fast'  # minimal reasoning - no LLM, fastest
KB_NAME        = 'arxiv-nlp-kb'       # low reasoning - one LLM planning pass
MCP_CONN       = 'arxiv-nlp-mcp'      # RemoteTool connection → arxiv-nlp-kb
MCP_CONN_FAST  = 'arxiv-nlp-mcp-fast' # RemoteTool connection → arxiv-nlp-kb-fast
AGENT_NAME     = 'arxiv-nlp-agent'

# ── Environment variables ────────────────────────────────────────────────────
search_endpoint   = os.environ['IQ_SEARCH_ENDPOINT']
project_endpoint  = os.environ['IQ_FOUNDRY_PROJECT_ENDPOINT']
apim_connection   = os.environ.get('IQ_APIM_CONNECTION', 'iq-apim-connection')
gateway_url       = os.environ['GATEWAY_URL']
iq_gateway_key    = os.environ['IQ_GATEWAY_KEY']
chat_model        = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

# AzureOpenAIVectorizerParameters.resource_url must be the root APIM URL (no /openai).
# The SDK/search service appends /openai/deployments/... automatically.
apim_base = gateway_url.rstrip('/').removesuffix('/openai')

# Foundry account endpoint (for the KB model config)
# Derived: https://<account>.services.ai.azure.com/api/projects/<project>
#       -> https://<account>.services.ai.azure.com
account_endpoint = project_endpoint.split('/api/projects/')[0]

# Agent model reference: <apim-connection>/<chat-model>
agent_model = f'{apim_connection}/{chat_model}'

# MCP endpoints exposed by each KB
MCP_ENDPOINT      = f'{search_endpoint}/knowledgebases/{KB_NAME}/mcp?api-version=2025-11-01-Preview'
MCP_ENDPOINT_FAST = f'{search_endpoint}/knowledgebases/{KB_FAST}/mcp?api-version=2025-11-01-Preview'

# Azure subscription and resource group (for MCP connection management API call)
subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID') or subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()

resource_group = os.environ.get('IQ_RESOURCE_GROUP', 'rg-foundry-iq')

# Extract account name and project name from endpoint
# e.g. https://iq-spoke-abc123.services.ai.azure.com/api/projects/iq-project
account_name  = account_endpoint.split('//')[1].split('.')[0]  # iq-spoke-abc123
project_name  = project_endpoint.split('/api/projects/')[1]    # iq-project

print(f'Search endpoint  : {search_endpoint}')
print(f'Project endpoint : {project_endpoint}')
print(f'Account name     : {account_name}')
print(f'Project name     : {project_name}')
print(f'APIM base        : {apim_base}')
print(f'Agent model      : {agent_model}')
print(f'MCP endpoint     : {MCP_ENDPOINT}')
print(f'MCP endpoint fast: {MCP_ENDPOINT_FAST}')
print(f'Subscription     : {subscription_id}')
print(f'Resource group   : {resource_group}')

Search endpoint  : https://iq-search-gvwiex.search.windows.net
Project endpoint : https://aif-spoke-multi-gvwiex.services.ai.azure.com/api/projects/iq-project
Account name     : aif-spoke-multi-gvwiex
Project name     : iq-project
APIM base        : https://apim-foundry-6fe574.azure-api.net
Agent model      : iq-apim-connection/gpt-4.1-mini
MCP endpoint     : https://iq-search-gvwiex.search.windows.net/knowledgebases/arxiv-nlp-kb/mcp?api-version=2025-11-01-Preview
MCP endpoint fast: https://iq-search-gvwiex.search.windows.net/knowledgebases/arxiv-nlp-kb-fast/mcp?api-version=2025-11-01-Preview
Subscription     : 00000000-0000-0000-0000-000000000000
Resource group   : rg-foundry-multi-6fe574


## Create clients

In [2]:
from azure.search.documents.indexes import SearchIndexClient
from azure.ai.projects import AIProjectClient

credential     = DefaultAzureCredential()
index_client   = SearchIndexClient(endpoint=search_endpoint, credential=credential)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)

print('Search index client : ready')
print('Project client      : ready')

Search index client : ready
Project client      : ready


---
## Phase 1: Knowledge source

A **knowledge source** registers an existing search index as a named retrieval target.
It specifies:
- Which index to search (`arxiv-nlp`)
- Which semantic configuration to apply (`arxiv-nlp-semantic`)
- Which fields to surface as **citation metadata** (human-readable only - no vectors)

`create_or_update_knowledge_source` is idempotent - safe to re-run.

In [3]:
# Assert the index exists before creating the knowledge source
indexes = [i.name for i in index_client.list_indexes()]
assert INDEX_NAME in indexes, (
    f"Index '{INDEX_NAME}' not found - run 10-02-index-and-ingest.ipynb first."
)
print(f"Index '{INDEX_NAME}' found.")

Index 'arxiv-nlp' found.


In [4]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
)

ks = SearchIndexKnowledgeSource(
    name=KS_NAME,
    description='NLP paper abstracts from arXiv (3,000 documents, 1994-2024)',
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        semantic_configuration_name='arxiv-nlp-semantic',
        source_data_fields=[
            # Citation metadata fields only: vector fields are intentionally excluded
            SearchIndexFieldReference(name='id'),
            SearchIndexFieldReference(name='title'),
            SearchIndexFieldReference(name='year'),
            SearchIndexFieldReference(name='categories'),
        ],
        # search_fields left empty -> all searchable fields are searched
    ),
)

index_client.create_or_update_knowledge_source(ks)
print(f"Knowledge source '{KS_NAME}' created.")

Knowledge source 'arxiv-nlp-ks' created.


---
## Phase 2: Knowledge bases

Two KB variants are created against the same knowledge source to demonstrate the
`retrieval_reasoning_effort` trade-off:

| KB | Effort | LLM | Use case |
|----|--------|-----|----------|
| `arxiv-nlp-kb-fast` | `minimal` | None | Simple lookups, low latency, no quota |
| `arxiv-nlp-kb` | `low` | `gpt-4.1-mini` via APIM | Complex/multi-turn questions, better relevance |

Both use `output_mode=EXTRACTIVE_DATA` - the KB returns raw chunks for the agent's
own LLM to reason over (as opposed to `ANSWER_SYNTHESIS` where the KB generates the
natural-language answer itself).

In [5]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
    AzureOpenAIVectorizerParameters,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
)

# ── KB variant 1: minimal reasoning: no LLM, direct index-driven retrieval ──
kb_fast = KnowledgeBase(
    name=KB_FAST,
    description='arxiv-nlp KB - minimal effort, no LLM planning, fastest response.',
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    knowledge_sources=[KnowledgeSourceReference(name=KS_NAME)],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    # No models[]: minimal effort does not use an LLM
)

index_client.create_or_update_knowledge_base(kb_fast)
print(f"Knowledge base '{KB_FAST}' created (minimal effort, no LLM).")

Knowledge base 'arxiv-nlp-kb-fast' created (minimal effort, no LLM).


In [6]:
# ── KB variant 2: low reasoning: one LLM pass for query planning via APIM ──
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=apim_base,
    deployment_name=chat_model,
    model_name=chat_model,
    api_key=iq_gateway_key,
)

kb = KnowledgeBase(
    name=KB_NAME,
    description='arxiv-nlp KB - low effort, LLM query planning via APIM gateway.',
    retrieval_instructions=(
        'Answer questions about NLP research papers using the arxiv-nlp knowledge source. '
        'Cite paper titles and publication years in your answers.'
    ),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    knowledge_sources=[KnowledgeSourceReference(name=KS_NAME)],
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
)

index_client.create_or_update_knowledge_base(kb)
print(f"Knowledge base '{KB_NAME}' created (low effort, LLM via APIM).")

Knowledge base 'arxiv-nlp-kb' created (low effort, LLM via APIM).


---
## Phase 3: Inline KB validation

Validate that each KB works **before** wiring it to an agent. This isolates retrieval
failures from agent/tool configuration issues.

`KnowledgeBaseRetrievalClient` calls the KB directly without going through an agent.

### `minimal` effort: intents, not messages

The `arxiv-nlp-kb-fast` KB uses `minimal` reasoning effort, which means **no LLM is
involved at any stage**.

The retrieval API accepts two input shapes:

- **`messages`** - a conversation history (same shape as a chat API call: `role: user`,
  `content: "..."`, optionally prior turns). The KB passes this to its internal LLM,
  which reads the conversation and decides what to search for. Requires a model -
  only valid for `low`/`medium`/`high` effort KBs.

- **`intents`** - a pre-parsed search directive: `KnowledgeRetrievalSemanticIntent(search="...")`.
  The KB executes this as a semantic search query directly, with no natural-language
  interpretation. The caller has already decided what to search for.

With `minimal` effort the KB has no LLM, so it cannot interpret a conversation -
hence `messages` is rejected and `intents` is required.

In the agent use case this is not a limitation: the agent's own LLM decides what to
search and calls the MCP tool with a query string. The MCP layer translates that into
an `intents` request automatically, so `minimal` effort works seamlessly as an agent
tool despite having no model of its own.

### `EXTRACTIVE_DATA` output mode: raw chunks, no composition

Both KBs are configured with `output_mode=EXTRACTIVE_DATA`. In this mode the KB
**never touches the content** - it returns the raw retrieved chunks exactly as they
were indexed. No summarisation, no rewriting, no synthesis. The text in each chunk
card is literally the abstract as it was ingested.

The reasoning effort level only influences **how the chunks were selected**, not what
format they come back in:

- **`minimal` + `EXTRACTIVE_DATA`** - pure index retrieval. The query string goes
  straight to Azure AI Search (BM25 + semantic ranker). Whatever the ranker returns
  is what you get. No LLM involved at any point.

- **`low` + `EXTRACTIVE_DATA`** - the KB's LLM reads the query first and can
  decompose it into multiple sub-queries, decide which knowledge sources to search,
  and rerank the merged results. It then hands the raw chunks straight back - content
  untouched. The LLM only influenced *which* chunks were selected.

For a simple single-concept query the two outputs will look nearly identical. The
difference becomes visible with complex multi-part questions where `low` effort
decomposes the query and returns more targeted chunks than a single literal search
would find.

### `EXTRACTIVE_DATA` vs `ANSWER_SYNTHESIS`

The output mode and reasoning effort are independent axes:

| | `EXTRACTIVE_DATA` | `ANSWER_SYNTHESIS` |
|---|---|---|
| **`minimal` effort** | Raw chunks, no LLM at all | ❌ Not supported |
| **`low` effort** | Raw chunks, LLM for query planning only | LLM for query planning **and** answer composition |

`EXTRACTIVE_DATA` is the right choice when the KB feeds an agent - the agent's own
LLM does the composing and citing. `ANSWER_SYNTHESIS` makes more sense for direct KB
queries where you want a finished natural-language answer without an agent in the loop.

In [7]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)
from display_helpers import show_kb_result_detail

VALIDATION_QUERY = 'What are the main approaches to transformer-based named entity recognition?'


def make_kb_request(query: str) -> KnowledgeBaseRetrievalRequest:
    """Request using messages - for low/medium/high reasoning effort KBs."""
    return KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role='user',
                content=[KnowledgeBaseMessageTextContent(text=query)],
            )
        ],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KS_NAME,
                include_references=True,
                include_reference_source_data=True,
            )
        ],
        include_activity=True,
    )


def make_kb_intent_request(query: str) -> KnowledgeBaseRetrievalRequest:
    """Request using intents - required for minimal reasoning effort KBs (no LLM)."""
    return KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=query)],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KS_NAME,
                include_references=True,
                include_reference_source_data=True,
            )
        ],
        include_activity=True,
    )


def refs_to_list(references):
    """Normalize references - preserve source_data so display helpers can show year, categories etc."""
    return [
        {
            'title':       (r.source_data or {}).get('title', r.id),
            'id':          str(r.id),
            'source_data': r.source_data or {},
        }
        for r in (references or [])
    ]


# Validate the fast KB (minimal effort: must use intents, not messages)
kb_fast_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KB_FAST,
    credential=credential,
)

result_fast = kb_fast_client.retrieve(make_kb_intent_request(VALIDATION_QUERY))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_fast.response[0].content[0].text}]}],
        'references': refs_to_list(result_fast.references),
    },
    label=f'{KB_FAST} - validation (minimal effort)',
)
print()

**arxiv-nlp-kb-fast — validation (minimal effort)**

**21 retrieved chunk(s):**

In [8]:
# Validate the standard KB (low effort)
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KB_NAME,
    credential=credential,
)

result_kb = kb_client.retrieve(make_kb_request(VALIDATION_QUERY))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_kb.response[0].content[0].text}]}],
        'references': refs_to_list(result_kb.references),
    },
    label=f'{KB_NAME} - validation (low effort)',
)

**arxiv-nlp-kb — validation (low effort)**

**21 retrieved chunk(s):**

---
## Phase 3b: Dual reasoning-effort comparison

Run the same query through both KBs side-by-side to directly observe the retrieval
trade-off between `minimal` and `low` effort.

| KB | Effort | Query input | What happens |
|----|--------|-------------|--------------|
| `arxiv-nlp-kb-fast` | `minimal` | `intents` | Direct semantic search - no LLM involved |
| `arxiv-nlp-kb` | `low` | `messages` | LLM decomposes query into sub-queries, fans out, merges and reranks |

On a simple factual query both KBs return similar results. On a complex multi-part
question the low-effort KB typically returns more relevant and diverse citations because
the LLM planning pass identifies the separate sub-topics and routes them independently.

In [9]:
COMPARISON_QUERY = (
    'What techniques are used for cross-lingual transfer learning in low-resource NLP?'
)

print(f'Query: {COMPARISON_QUERY!r}\n')

# minimal: intents, direct retrieval, no LLM
result_minimal = kb_fast_client.retrieve(make_kb_intent_request(COMPARISON_QUERY))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_minimal.response[0].content[0].text}]}],
        'references': refs_to_list(result_minimal.references),
    },
    label=f'{KB_FAST} - minimal effort (no LLM, direct retrieval)',
)
print()

# low: messages, one LLM planning pass for query decomposition
result_low = kb_client.retrieve(make_kb_request(COMPARISON_QUERY))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_low.response[0].content[0].text}]}],
        'references': refs_to_list(result_low.references),
    },
    label=f'{KB_NAME} - low effort (LLM query planning via APIM)',
)

Query: 'What techniques are used for cross-lingual transfer learning in low-resource NLP?'



**arxiv-nlp-kb-fast — minimal effort (no LLM, direct retrieval)**

**23 retrieved chunk(s):**

**arxiv-nlp-kb — low effort (LLM query planning via APIM)**

**24 retrieved chunk(s):**

## Phase 3c: Security trimming: `filterAddOn`

The `group_ids/any(...)` OData filter can be injected into a KB retrieval via
`filterAddOn` on `knowledgeSourceParams`. This enforces per-document access control
at the search layer - not the application layer - making it tamper-resistant.

The filter is exactly the same OData expression used in 10-04-search-patterns Pattern 6,
but here it is applied through the Foundry IQ KB retrieval API rather than directly
against Azure AI Search.

In [10]:
# "machine translation" spans pre- and post-2010 papers in the index.
# The semantic ranker tends to surface post-2010 papers regardless of the group filter,
# so we add a third call that forces year < 2010 to confirm archive access explicitly.
SECURITY_QUERY = 'machine translation approaches and methods'


def make_kb_intent_request_filtered(query: str, filter_add_on: str) -> KnowledgeBaseRetrievalRequest:
    """Intent request with an arbitrary OData filterAddOn."""
    return KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=query)],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KS_NAME,
                include_references=True,
                include_reference_source_data=True,
                filter_add_on=filter_add_on,
            )
        ],
        include_activity=True,
    )


# 1. Public caller: only documents tagged 'public' → 2010+ papers
result_public = kb_fast_client.retrieve(make_kb_intent_request_filtered(
    SECURITY_QUERY,
    filter_add_on="group_ids/any(g: search.in(g, 'public'))",
))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_public.response[0].content[0].text}]}],
        'references': refs_to_list(result_public.references),
    },
    label="public caller - expect: all years ≥ 2010",
)
print()

# 2. Archive caller: all documents tagged 'archive' → all years
result_archive = kb_fast_client.retrieve(make_kb_intent_request_filtered(
    SECURITY_QUERY,
    filter_add_on="group_ids/any(g: search.in(g, 'archive'))",
))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_archive.response[0].content[0].text}]}],
        'references': refs_to_list(result_archive.references),
    },
    label="archive caller - all years (semantic ranker may still favour recent papers)",
)
print()

# 3. Archive caller, forced pre-2010: proves archive group can access older documents
#    that the public group cannot. Combines the group filter with an explicit year constraint.
result_archive_old = kb_fast_client.retrieve(make_kb_intent_request_filtered(
    SECURITY_QUERY,
    filter_add_on="group_ids/any(g: search.in(g, 'archive')) and year lt 2010",
))
show_kb_result_detail(
    {
        'response': [{'content': [{'text': result_archive_old.response[0].content[0].text}]}],
        'references': refs_to_list(result_archive_old.references),
    },
    label="archive caller (pre-2010 only) - confirms access to documents the public group cannot see",
)

**public caller — expect: all years ≥ 2010**

**19 retrieved chunk(s):**

**archive caller — all years (semantic ranker may still favour recent papers)**

**19 retrieved chunk(s):**

**archive caller (pre-2010 only) — confirms access to documents the public group cannot see**

**19 retrieved chunk(s):**

---
## Phase 4: MCP Connections

Each knowledge base automatically exposes an MCP endpoint:
```
{search_endpoint}/knowledgebases/{name}/mcp?api-version=2025-11-01-Preview
```

To attach these endpoints to a Foundry agent as tools, the Foundry project needs a
**RemoteTool** project connection for each KB. Connections are created **explicitly**
via the Azure Management REST API - no portal interaction required.

| Connection | KB | Effort |
|------------|----|--------|
| `arxiv-nlp-mcp` | `arxiv-nlp-kb` | `low` - LLM query planning |
| `arxiv-nlp-mcp-fast` | `arxiv-nlp-kb-fast` | `minimal` - direct retrieval |

| Property | Value |
|----------|-------|
| `category` | `RemoteTool` |
| `authType` | `ProjectManagedIdentity` |
| `audience` | `https://search.azure.com/` |
| `target` | MCP endpoint URL |

> The project managed identity has **Search Index Data Reader** on the search service
> from the Bicep deployment, which is what `ProjectManagedIdentity` auth uses to
> authenticate to the MCP endpoint.

In [11]:
from iq_helpers import create_mcp_connection
from display_helpers import show_success, show_error

# ── Standard KB connection (low effort) ──────────────────────────────────────
result = create_mcp_connection(
    subscription_id=subscription_id,
    resource_group=resource_group,
    account_name=account_name,
    project_name=project_name,
    connection_name=MCP_CONN,
    search_endpoint=search_endpoint,
    kb_name=KB_NAME,
)
if 'error' not in result:
    show_success(f"MCP connection '{MCP_CONN}' created.")
    print(f'  Endpoint: {MCP_ENDPOINT}')
else:
    show_error(str(result))

print()

# ── Fast KB connection (minimal effort) ──────────────────────────────────────
result_fast = create_mcp_connection(
    subscription_id=subscription_id,
    resource_group=resource_group,
    account_name=account_name,
    project_name=project_name,
    connection_name=MCP_CONN_FAST,
    search_endpoint=search_endpoint,
    kb_name=KB_FAST,
)
if 'error' not in result_fast:
    show_success(f"MCP connection '{MCP_CONN_FAST}' created.")
    print(f'  Endpoint: {MCP_ENDPOINT_FAST}')
else:
    show_error(str(result_fast))

### ✅ MCP connection 'arxiv-nlp-mcp' created.

  Endpoint: https://iq-search-gvwiex.search.windows.net/knowledgebases/arxiv-nlp-kb/mcp?api-version=2025-11-01-Preview



### ✅ MCP connection 'arxiv-nlp-mcp-fast' created.

  Endpoint: https://iq-search-gvwiex.search.windows.net/knowledgebases/arxiv-nlp-kb-fast/mcp?api-version=2025-11-01-Preview


---
## Phase 5: Versioned Agent

`create_version()` creates the agent on first run. On subsequent runs it compares the
`PromptAgentDefinition` to the latest stored version and only increments the version
when something has changed - safe to re-run while iterating on instructions.

The agent is equipped with **two MCPTool instances**, one per KB. The agent's own LLM
decides which tool to call based on the instructions and query complexity:

| Tool label | KB | When the agent uses it |
|------------|----|------------------------|
| `kb_fast` | `arxiv-nlp-kb-fast` | Simple factual lookups, single-topic questions |
| `kb_standard` | `arxiv-nlp-kb` | Complex, multi-part, or analytical questions |

The version number is written to `.env` as `IQ_AGENT_VERSION` and used in
`10-05-agent-iq-queries.ipynb` to pin to this exact definition.

In [12]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from dotenv import set_key

instructions = """
You are a research assistant specialising in NLP papers from arXiv (1994-2024).

You have two retrieval tools:
- kb_fast: fast, direct retrieval - use for simple, single-topic factual questions
- kb_standard: LLM-planned retrieval - use for complex, multi-part, or analytical questions

Rules:
- Always use one of the retrieval tools to answer questions. Never answer from your own training data.
- For each claim, cite the paper title and year in parentheses: (Title, Year).
- If the answer is not in the knowledge base, respond with exactly: "I don't know."
- Do not speculate or extrapolate beyond what the knowledge base returns.
"""

mcp_tool_fast = MCPTool(
    server_label='kb_fast',
    server_url=MCP_ENDPOINT_FAST,
    require_approval='never',
    allowed_tools=['knowledge_base_retrieve'],
    project_connection_id=MCP_CONN_FAST,
)

mcp_tool_standard = MCPTool(
    server_label='kb_standard',
    server_url=MCP_ENDPOINT,
    require_approval='never',
    allowed_tools=['knowledge_base_retrieve'],
    project_connection_id=MCP_CONN,
)

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=agent_model,
        instructions=instructions,
        tools=[mcp_tool_fast, mcp_tool_standard],
    ),
    description='Research assistant over arXiv NLP papers - dual KB tools (minimal + low effort).',
)

AGENT_VERSION = agent.version
print(f"Agent '{AGENT_NAME}' created (version {AGENT_VERSION}).")
print(f'Model         : {agent_model}')
print(f'Tools         : kb_fast ({KB_FAST}), kb_standard ({KB_NAME})')
print()

# Persist version to .env so 10-05 can pin to it
set_key(str(repo_root / '.env'), 'IQ_AGENT_VERSION', str(AGENT_VERSION))
print(f'IQ_AGENT_VERSION={AGENT_VERSION} written to .env')

Agent 'arxiv-nlp-agent' created (version 2).
Model         : iq-apim-connection/gpt-4.1-mini
Tools         : kb_fast (arxiv-nlp-kb-fast), kb_standard (arxiv-nlp-kb)

IQ_AGENT_VERSION=2 written to .env


---
## Cleanup *(optional)*

Run the cells below to remove all resources created by this notebook.
Delete in order: agent → MCP connections → KBs → knowledge source.

In [13]:
# project_client.agents.delete_version(AGENT_NAME, AGENT_VERSION)
# print(f"Agent '{AGENT_NAME}' v{AGENT_VERSION} deleted.")

# import requests
# from iq_helpers import get_mgmt_token
# base = (f"https://management.azure.com/subscriptions/{subscription_id}"
#         f"/resourceGroups/{resource_group}"
#         f"/providers/Microsoft.CognitiveServices/accounts/{account_name}"
#         f"/projects/{project_name}/connections")
# hdrs = {'Authorization': f'Bearer {get_mgmt_token()}', 'Content-Type': 'application/json'}
# for conn in [MCP_CONN, MCP_CONN_FAST]:
#     r = requests.delete(f"{base}/{conn}?api-version=2025-04-01-preview", headers=hdrs)
#     print(f"Connection '{conn}' deleted: {r.status_code}")

# index_client.delete_knowledge_base(KB_NAME)
# print(f"Knowledge base '{KB_NAME}' deleted.")
# index_client.delete_knowledge_base(KB_FAST)
# print(f"Knowledge base '{KB_FAST}' deleted.")
# index_client.delete_knowledge_source(KS_NAME)
# print(f"Knowledge source '{KS_NAME}' deleted.")